In [1]:
from google.cloud import storage
import pandas as pd
import joblib
import json
from sklearn.metrics import accuracy_score
import os
from feast import FeatureStore
import mlflow.pyfunc

/home/athipse/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
store = FeatureStore(repo_path="feature_repo")

OUTPUT_BUCKET = "mlops-course-week1-unique"

storage_client = storage.Client()

artifact_bucket = storage_client.bucket(OUTPUT_BUCKET)

In [3]:
prefixes = sorted(
    set(
        blob.name.split("/")[0]
        for blob in artifact_bucket.list_blobs()
    )
)

latest_run = prefixes[-1]

In [5]:
model = mlflow.pyfunc.load_model(
    "models:/IrisClassifier/Production"
)

MlflowException: No versions of model with name 'IrisClassifier' and stage 'Production' found

In [ ]:
online_features = store.get_online_features(

    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],

    entity_rows=[
        {
            "iris_id": 50
        }
    ]

).to_dict()

In [ ]:
X = pd.DataFrame({

    "sepal_length":[online_features["sepal_length"][0]],

    "sepal_width":[online_features["sepal_width"][0]],

    "petal_length":[online_features["petal_length"][0]],

    "petal_width":[online_features["petal_width"][0]]

})

preds = model.predict(X)
print(X)
print(preds)

   sepal_length  sepal_width  petal_length  petal_width
0           7.0          3.2           4.7          1.4
['versicolor']


## Verify prediction

In [ ]:
raw = pd.read_parquet("feature_repo/data/iris.parquet")

print(raw.loc[50])

sepal_length                              7.0
sepal_width                               3.2
petal_length                              4.7
petal_width                               1.4
species                            versicolor
iris_id                                    50
event_timestamp    2026-07-05 07:34:18.674435
Name: 50, dtype: object
